# Water Segmentation with U-Net
This notebook implements a U-Net model for water body segmentation in satellite/aerial imagery.

**Key Features:**
- U-Net architecture with residual connections
- Adaptive learning rate scheduling (ReduceLROnPlateau)
- Mixed loss function (BCE + Dice)
- Comprehensive metrics tracking
- Best model checkpointing

# Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install Dependencies

In [ ]:
!apt-get install -y python3-gdal
!pip install rasterio geopandas tqdm matplotlib scipy albumentations segmentation-models-pytorch

# Verify installation
import rasterio
import geopandas as gpd
import torch

print(f"rasterio version: {rasterio.__version__}")
print(f"geopandas version: {gpd.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("✅ All packages installed successfully!")

# Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Geospatial libraries
import rasterio
import geopandas as gpd

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Image augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Metrics

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Configuration

In [ ]:
class Config:
    """Configuration class for training parameters"""
    
    # Paths (update these according to your setup)
    DATA_DIR = "/content/drive/MyDrive/water_segmentation/data"
    OUTPUT_DIR = "/content/drive/MyDrive/water_segmentation/output"
    TRAIN_IMAGES = os.path.join(DATA_DIR, "train/images")
    TRAIN_MASKS = os.path.join(DATA_DIR, "train/masks")
    VAL_IMAGES = os.path.join(DATA_DIR, "val/images")
    VAL_MASKS = os.path.join(DATA_DIR, "val/masks")
    TEST_IMAGES = os.path.join(DATA_DIR, "test/images")
    GROUND_TRUTH = os.path.join(DATA_DIR, "test/masks")
    
    # Model parameters
    IMG_SIZE = 256
    IN_CHANNELS = 3
    OUT_CHANNELS = 1
    INIT_FEATURES = 64
    
    # Training parameters
    BATCH_SIZE = 8
    NUM_EPOCHS = 100
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 1e-5
    
    # Learning rate scheduler parameters
    LR_PATIENCE = 5  # Number of epochs with no improvement after which LR will be reduced
    LR_FACTOR = 0.5  # Factor by which the learning rate will be reduced
    LR_MIN = 1e-7    # Minimum learning rate
    
    # Early stopping
    EARLY_STOP_PATIENCE = 15
    
    # Mixed precision training
    USE_AMP = True
    
    # Data augmentation
    AUGMENT = True
    
    # Checkpoint
    SAVE_BEST_ONLY = True
    
config = Config()

# Create output directories
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(config.OUTPUT_DIR, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(config.OUTPUT_DIR, "metrics"), exist_ok=True)
os.makedirs(os.path.join(config.OUTPUT_DIR, "predictions"), exist_ok=True)

print("Configuration loaded successfully!")
print(f"Output directory: {config.OUTPUT_DIR}")

# Data Augmentation

In [ ]:
def get_training_augmentation():
    """Returns augmentation pipeline for training"""
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 50.0), p=1),
            A.GaussianBlur(blur_limit=(3, 7), p=1),
        ], p=0.3),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1),
            A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=1),
        ], p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_validation_augmentation():
    """Returns augmentation pipeline for validation/testing"""
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

print("Augmentation pipelines defined!")

# Dataset Class

In [ ]:
class WaterSegmentationDataset(Dataset):
    """Dataset for water segmentation task"""
    
    def __init__(self, image_dir, mask_dir, transform=None):
        """
        Args:
            image_dir: Directory containing input images
            mask_dir: Directory containing mask images
            transform: Albumentations transform pipeline
        """
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        
        # Get list of image files
        self.images = sorted([f for f in os.listdir(image_dir) if f.endswith(('.tif', '.tiff', '.png', '.jpg'))])
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        # Load image
        img_name = self.images[idx]
        img_path = os.path.join(self.image_dir, img_name)
        
        # Handle different mask naming conventions
        base_name = os.path.splitext(img_name)[0]
        mask_name = f"{base_name}_mask.tif"
        if not os.path.exists(os.path.join(self.mask_dir, mask_name)):
            mask_name = img_name
        
        mask_path = os.path.join(self.mask_dir, mask_name)
        
        # Read image and mask using rasterio
        with rasterio.open(img_path) as src:
            image = src.read([1, 2, 3]).transpose(1, 2, 0)  # Read RGB bands
            image = image.astype(np.float32)
            
            # Normalize to [0, 1] if needed
            if image.max() > 1.0:
                image = image / 255.0
        
        with rasterio.open(mask_path) as src:
            mask = src.read(1).astype(np.float32)
            mask = (mask > 0).astype(np.float32)  # Binarize mask
        
        # Apply transformations
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']
        
        # Add channel dimension to mask
        mask = mask.unsqueeze(0) if len(mask.shape) == 2 else mask
        
        return image, mask

print("Dataset class defined!")

# U-Net Model Architecture

In [ ]:
class DoubleConv(nn.Module):
    """Double convolution block with batch normalization"""
    
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if mid_channels is None:
            mid_channels = out_channels
        
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """Downscaling with maxpool then double conv"""
    
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    
    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """Upscaling then double conv"""
    
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)
    
    def forward(self, x1, x2):
        x1 = self.up(x1)
        
        # Handle size mismatch
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    """Output convolution layer"""
    
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
    
    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    """U-Net architecture for image segmentation"""
    
    def __init__(self, n_channels=3, n_classes=1, init_features=64, bilinear=False):
        super().__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear
        
        self.inc = DoubleConv(n_channels, init_features)
        self.down1 = Down(init_features, init_features * 2)
        self.down2 = Down(init_features * 2, init_features * 4)
        self.down3 = Down(init_features * 4, init_features * 8)
        factor = 2 if bilinear else 1
        self.down4 = Down(init_features * 8, init_features * 16 // factor)
        
        self.up1 = Up(init_features * 16, init_features * 8 // factor, bilinear)
        self.up2 = Up(init_features * 8, init_features * 4 // factor, bilinear)
        self.up3 = Up(init_features * 4, init_features * 2 // factor, bilinear)
        self.up4 = Up(init_features * 2, init_features, bilinear)
        self.outc = OutConv(init_features, n_classes)
    
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

print("U-Net model defined!")

# Loss Functions

In [ ]:
class DiceLoss(nn.Module):
    """Dice loss for binary segmentation"""
    
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, predictions, targets):
        predictions = torch.sigmoid(predictions)
        
        # Flatten
        predictions = predictions.view(-1)
        targets = targets.view(-1)
        
        intersection = (predictions * targets).sum()
        dice = (2. * intersection + self.smooth) / (predictions.sum() + targets.sum() + self.smooth)
        
        return 1 - dice


class BCEDiceLoss(nn.Module):
    """Combined Binary Cross Entropy and Dice Loss"""
    
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
    
    def forward(self, predictions, targets):
        bce_loss = self.bce(predictions, targets)
        dice_loss = self.dice(predictions, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

print("Loss functions defined!")

# Metrics Calculation

In [ ]:
def calculate_metrics(predictions, targets, threshold=0.5):
    """
    Calculate comprehensive segmentation metrics
    
    Args:
        predictions: Model predictions (logits or probabilities)
        targets: Ground truth masks
        threshold: Threshold for binary classification
    
    Returns:
        Dictionary of metrics
    """
    # Apply sigmoid if predictions are logits
    if predictions.min() < 0 or predictions.max() > 1:
        predictions = torch.sigmoid(predictions)
    
    # Binarize predictions
    preds_binary = (predictions > threshold).float()
    
    # Flatten tensors
    preds_flat = preds_binary.view(-1)
    targets_flat = targets.view(-1)
    
    # Calculate confusion matrix components
    tp = (preds_flat * targets_flat).sum()
    fp = (preds_flat * (1 - targets_flat)).sum()
    fn = ((1 - preds_flat) * targets_flat).sum()
    tn = ((1 - preds_flat) * (1 - targets_flat)).sum()
    
    # Calculate metrics
    smooth = 1e-7
    
    # IoU (Jaccard Index)
    iou = (tp + smooth) / (tp + fp + fn + smooth)
    
    # Dice Score
    dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    
    # Precision
    precision = (tp + smooth) / (tp + fp + smooth)
    
    # Recall (Sensitivity)
    recall = (tp + smooth) / (tp + fn + smooth)
    
    # Specificity
    specificity = (tn + smooth) / (tn + fp + smooth)
    
    # Accuracy
    accuracy = (tp + tn + smooth) / (tp + tn + fp + fn + smooth)
    
    # F1 Score
    f1 = (2 * precision * recall) / (precision + recall + smooth)
    
    return {
        'iou': iou.item(),
        'dice': dice.item(),
        'precision': precision.item(),
        'recall': recall.item(),
        'specificity': specificity.item(),
        'accuracy': accuracy.item(),
        'f1_score': f1.item()
    }

print("Metrics calculation function defined!")

# Training Functions

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, scaler=None):
    """
    Train model for one epoch
    
    Args:
        model: PyTorch model
        dataloader: Training dataloader
        criterion: Loss function
        optimizer: Optimizer
        device: Device to train on
        scaler: GradScaler for mixed precision training
    
    Returns:
        Average loss and metrics for the epoch
    """
    model.train()
    running_loss = 0.0
    all_metrics = {k: 0.0 for k in ['iou', 'dice', 'precision', 'recall', 'specificity', 'accuracy', 'f1_score']}
    
    pbar = tqdm(dataloader, desc="Training")
    for images, masks in pbar:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        
        # Mixed precision training
        if scaler is not None:
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, masks)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
        
        # Calculate metrics
        with torch.no_grad():
            batch_metrics = calculate_metrics(outputs, masks)
            for k in all_metrics:
                all_metrics[k] += batch_metrics[k]
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': loss.item(), 'dice': batch_metrics['dice']})
    
    epoch_loss = running_loss / len(dataloader)
    epoch_metrics = {k: v / len(dataloader) for k, v in all_metrics.items()}
    
    return epoch_loss, epoch_metrics


def validate(model, dataloader, criterion, device):
    """
    Validate model
    
    Args:
        model: PyTorch model
        dataloader: Validation dataloader
        criterion: Loss function
        device: Device to validate on
    
    Returns:
        Average loss and metrics for validation
    """
    model.eval()
    running_loss = 0.0
    all_metrics = {k: 0.0 for k in ['iou', 'dice', 'precision', 'recall', 'specificity', 'accuracy', 'f1_score']}
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc="Validation")
        for images, masks in pbar:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            # Calculate metrics
            batch_metrics = calculate_metrics(outputs, masks)
            for k in all_metrics:
                all_metrics[k] += batch_metrics[k]
            
            running_loss += loss.item()
            pbar.set_postfix({'loss': loss.item(), 'dice': batch_metrics['dice']})
    
    epoch_loss = running_loss / len(dataloader)
    epoch_metrics = {k: v / len(dataloader) for k, v in all_metrics.items()}
    
    return epoch_loss, epoch_metrics

print("Training functions defined!")

# Create Data Loaders

In [ ]:
# Create datasets
train_dataset = WaterSegmentationDataset(
    config.TRAIN_IMAGES,
    config.TRAIN_MASKS,
    transform=get_training_augmentation() if config.AUGMENT else get_validation_augmentation()
)

val_dataset = WaterSegmentationDataset(
    config.VAL_IMAGES,
    config.VAL_MASKS,
    transform=get_validation_augmentation()
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Batch size: {config.BATCH_SIZE}")
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

# Initialize Model and Training Components

In [ ]:
# Initialize model
model = UNet(
    n_channels=config.IN_CHANNELS,
    n_classes=config.OUT_CHANNELS,
    init_features=config.INIT_FEATURES,
    bilinear=False
).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Initialize loss function
criterion = BCEDiceLoss(bce_weight=0.5, dice_weight=0.5)

# Initialize optimizer
optimizer = Adam(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

# Initialize learning rate scheduler with ReduceLROnPlateau
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',  # Minimize validation loss
    factor=config.LR_FACTOR,  # Multiply LR by this factor
    patience=config.LR_PATIENCE,  # Number of epochs with no improvement
    verbose=True,  # Print message when LR is reduced
    min_lr=config.LR_MIN  # Minimum learning rate
)

# Initialize gradient scaler for mixed precision
scaler = torch.cuda.amp.GradScaler() if config.USE_AMP and torch.cuda.is_available() else None

print("\n" + "="*60)
print("TRAINING CONFIGURATION")
print("="*60)
print(f"Device: {DEVICE}")
print(f"Mixed Precision: {config.USE_AMP and torch.cuda.is_available()}")
print(f"Learning Rate: {config.LEARNING_RATE}")
print(f"Batch Size: {config.BATCH_SIZE}")
print(f"Epochs: {config.NUM_EPOCHS}")
print("LR Scheduler: ReduceLROnPlateau")
print(f"  - Patience: {config.LR_PATIENCE} epochs")
print(f"  - Factor: {config.LR_FACTOR}")
print(f"  - Min LR: {config.LR_MIN}")
print(f"Early Stopping Patience: {config.EARLY_STOP_PATIENCE} epochs")
print("="*60 + "\n")

# Training Loop

In [ ]:
# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'train_dice': [],
    'val_dice': [],
    'train_iou': [],
    'val_iou': [],
    'learning_rates': []
}

# Best model tracking
best_val_loss = float('inf')
best_val_dice = 0.0
epochs_without_improvement = 0

# Checkpoint paths
best_model_path = os.path.join(config.OUTPUT_DIR, "checkpoints", "best_model.pth")
final_model_path = os.path.join(config.OUTPUT_DIR, "checkpoints", "final_model.pth")

print("Starting training...\n")

for epoch in range(config.NUM_EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{config.NUM_EPOCHS}")
    print(f"{'='*60}")
    
    # Get current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Learning Rate: {current_lr:.2e}")
    history['learning_rates'].append(current_lr)
    
    # Train
    train_loss, train_metrics = train_one_epoch(
        model, train_loader, criterion, optimizer, DEVICE, scaler
    )
    
    # Validate
    val_loss, val_metrics = validate(model, val_loader, criterion, DEVICE)
    
    # Update learning rate scheduler based on validation loss
    scheduler.step(val_loss)
    
    # Store history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_dice'].append(train_metrics['dice'])
    history['val_dice'].append(val_metrics['dice'])
    history['train_iou'].append(train_metrics['iou'])
    history['val_iou'].append(val_metrics['iou'])
    
    # Print metrics
    print(f"\nTrain Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"Train Dice: {train_metrics['dice']:.4f} | Val Dice: {val_metrics['dice']:.4f}")
    print(f"Train IoU:  {train_metrics['iou']:.4f} | Val IoU:  {val_metrics['iou']:.4f}")
    
    # Check for improvement
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_dice = val_metrics['dice']
        epochs_without_improvement = 0
        
        # Save best model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_loss': val_loss,
            'val_dice': val_metrics['dice'],
            'config': config.__dict__
        }, best_model_path)
        
        print(f"\n✅ New best model saved! Val Loss: {val_loss:.4f} | Val Dice: {val_metrics['dice']:.4f}")
    else:
        epochs_without_improvement += 1
        print(f"\n⏳ No improvement for {epochs_without_improvement} epoch(s)")
    
    # Early stopping
    if epochs_without_improvement >= config.EARLY_STOP_PATIENCE:
        print(f"\n🛑 Early stopping triggered after {epoch + 1} epochs")
        print(f"Best Val Loss: {best_val_loss:.4f} | Best Val Dice: {best_val_dice:.4f}")
        break

# Save final model
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'val_loss': val_loss,
    'val_dice': val_metrics['dice'],
    'config': config.__dict__
}, final_model_path)

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"Best Val Loss: {best_val_loss:.4f}")
print(f"Best Val Dice: {best_val_dice:.4f}")
print(f"Models saved to: {os.path.join(config.OUTPUT_DIR, 'checkpoints')}")

# Plot Training History

In [ ]:
# Create plots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Loss plot
axes[0, 0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Dice score plot
axes[0, 1].plot(history['train_dice'], label='Train Dice', linewidth=2)
axes[0, 1].plot(history['val_dice'], label='Val Dice', linewidth=2)
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Dice Score', fontsize=12)
axes[0, 1].set_title('Training and Validation Dice Score', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# IoU plot
axes[1, 0].plot(history['train_iou'], label='Train IoU', linewidth=2)
axes[1, 0].plot(history['val_iou'], label='Val IoU', linewidth=2)
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('IoU', fontsize=12)
axes[1, 0].set_title('Training and Validation IoU', fontsize=14, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Learning rate plot
axes[1, 1].plot(history['learning_rates'], linewidth=2, color='red')
axes[1, 1].set_xlabel('Epoch', fontsize=12)
axes[1, 1].set_ylabel('Learning Rate', fontsize=12)
axes[1, 1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Training Metrics', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_DIR, "metrics", "training_history.png"), dpi=150, bbox_inches='tight')
plt.show()

# Save history to CSV
history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(config.OUTPUT_DIR, "metrics", "training_history.csv"), index=False)
print(f"\n✅ Training history saved to: {os.path.join(config.OUTPUT_DIR, 'metrics')}")

# Prediction and Evaluation

In [ ]:
class WaterSegmentationPredictor:
    """Predictor class for water segmentation"""
    
    def __init__(self, model_path, device=DEVICE):
        self.device = device
        
        # Load model
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)
        
        self.model = UNet(
            n_channels=config.IN_CHANNELS,
            n_classes=config.OUT_CHANNELS,
            init_features=config.INIT_FEATURES,
            bilinear=False
        ).to(device)
        
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()
        
        self.transform = get_validation_augmentation()
    
    def predict(self, image_path, threshold=0.5):
        """
        Predict water mask for an image
        
        Args:
            image_path: Path to input image
            threshold: Threshold for binary classification
        
        Returns:
            Binary mask, probability map, and image profile
        """
        # Read image
        with rasterio.open(image_path) as src:
            image = src.read([1, 2, 3]).transpose(1, 2, 0)
            image = image.astype(np.float32)
            
            if image.max() > 1.0:
                image = image / 255.0
            
            profile = src.profile
        
        # Apply transformations
        transformed = self.transform(image=image)
        image_tensor = transformed['image'].unsqueeze(0).to(self.device)
        
        # Predict
        with torch.no_grad():
            output = self.model(image_tensor)
            prob = torch.sigmoid(output).cpu().numpy()[0, 0]
            mask = (prob > threshold).astype(np.uint8)
        
        return mask, prob, profile

print("Predictor class defined!")

# Evaluate on Test Set

In [ ]:
# Load best and final models
best_predictor = WaterSegmentationPredictor(best_model_path)
final_predictor = WaterSegmentationPredictor(final_model_path)

# Get test images
test_images = sorted([os.path.join(config.TEST_IMAGES, f) 
                      for f in os.listdir(config.TEST_IMAGES) 
                      if f.endswith(('.tif', '.tiff', '.png', '.jpg'))])

print(f"Found {len(test_images)} test images")

# Function to evaluate model
def evaluate_model(predictor, test_images, ground_truth_folder, model_name):
    """Evaluate model on test set"""
    results = []
    
    for img_path in tqdm(test_images, desc=f"Evaluating {model_name}"):
        # Predict
        mask, prob, profile = predictor.predict(img_path)
        
        # Load ground truth
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        gt_path = os.path.join(ground_truth_folder, f"{base_name}_mask.tif")
        
        if os.path.exists(gt_path):
            with rasterio.open(gt_path) as src:
                gt_mask = src.read(1)
            gt_mask = (gt_mask > 0).astype(np.float32)
            
            # Convert to tensors for metric calculation
            pred_tensor = torch.from_numpy(mask.astype(np.float32)).unsqueeze(0).unsqueeze(0)
            gt_tensor = torch.from_numpy(gt_mask).unsqueeze(0).unsqueeze(0)
            
            # Calculate metrics
            metrics = calculate_metrics(pred_tensor, gt_tensor)
            
            results.append({
                'image': base_name,
                'model': model_name,
                **metrics,
                'water_percentage': (mask.sum() / mask.size) * 100,
                'gt_water_percentage': (gt_mask.sum() / gt_mask.size) * 100
            })
    
    return pd.DataFrame(results)

# Evaluate models
if os.path.exists(config.GROUND_TRUTH):
    print("\n" + "="*60)
    print("MODEL EVALUATION")
    print("="*60)
    
    best_results = evaluate_model(best_predictor, test_images, config.GROUND_TRUTH, "Best Model")
    final_results = evaluate_model(final_predictor, test_images, config.GROUND_TRUTH, "Final Model")
    
    # Combine results
    all_results = pd.concat([best_results, final_results], ignore_index=True)
    
    # Save results
    all_results.to_csv(os.path.join(config.OUTPUT_DIR, "metrics", "test_results.csv"), index=False)
    
    # Summary statistics
    summary = all_results.groupby('model').agg({
        'iou': ['mean', 'std', 'min', 'max'],
        'dice': ['mean', 'std', 'min', 'max'],
        'precision': ['mean', 'std'],
        'recall': ['mean', 'std'],
        'f1_score': ['mean', 'std']
    }).round(4)
    
    summary.to_csv(os.path.join(config.OUTPUT_DIR, "metrics", "test_summary.csv"))
    
    print("\n" + "="*60)
    print("TEST RESULTS SUMMARY")
    print("="*60)
    print(summary)
    print(f"\n✅ Results saved to: {os.path.join(config.OUTPUT_DIR, 'metrics')}")
else:
    print("\n⚠️ Ground truth folder not found. Skipping evaluation.")

# Visualize Sample Predictions

In [ ]:
# Visualize predictions for first 5 test images
num_samples = min(5, len(test_images))

for idx in range(num_samples):
    img_path = test_images[idx]
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    
    # Read original image
    with rasterio.open(img_path) as src:
        original = src.read([1, 2, 3]).transpose(1, 2, 0)
        if original.max() > 1.0:
            original = original / 255.0
    
    # Get predictions
    best_mask, best_prob, _ = best_predictor.predict(img_path)
    final_mask, final_prob, _ = final_predictor.predict(img_path)
    
    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Original image
    axes[0, 0].imshow(original)
    axes[0, 0].set_title('Original Image', fontsize=12, fontweight='bold')
    axes[0, 0].axis('off')
    
    # Best model probability
    im1 = axes[0, 1].imshow(best_prob, cmap='Blues', vmin=0, vmax=1)
    axes[0, 1].set_title('Best Model - Probability', fontsize=12, fontweight='bold')
    axes[0, 1].axis('off')
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046)
    
    # Best model mask
    axes[0, 2].imshow(best_mask, cmap='Blues')
    axes[0, 2].set_title(f'Best Model - Mask ({(best_mask.sum()/best_mask.size)*100:.1f}% water)', 
                         fontsize=12, fontweight='bold')
    axes[0, 2].axis('off')
    
    # Ground truth (if available)
    gt_path = os.path.join(config.GROUND_TRUTH, f"{base_name}_mask.tif")
    if os.path.exists(gt_path):
        with rasterio.open(gt_path) as src:
            gt_mask = src.read(1)
        gt_mask = (gt_mask > 0).astype(np.uint8)
        axes[1, 0].imshow(gt_mask, cmap='Blues')
        axes[1, 0].set_title(f'Ground Truth ({(gt_mask.sum()/gt_mask.size)*100:.1f}% water)', 
                            fontsize=12, fontweight='bold')
    else:
        axes[1, 0].text(0.5, 0.5, 'No Ground Truth', ha='center', va='center', fontsize=14)
    axes[1, 0].axis('off')
    
    # Final model probability
    im2 = axes[1, 1].imshow(final_prob, cmap='Blues', vmin=0, vmax=1)
    axes[1, 1].set_title('Final Model - Probability', fontsize=12, fontweight='bold')
    axes[1, 1].axis('off')
    plt.colorbar(im2, ax=axes[1, 1], fraction=0.046)
    
    # Final model mask
    axes[1, 2].imshow(final_mask, cmap='Blues')
    axes[1, 2].set_title(f'Final Model - Mask ({(final_mask.sum()/final_mask.size)*100:.1f}% water)', 
                         fontsize=12, fontweight='bold')
    axes[1, 2].axis('off')
    
    plt.suptitle(f'Predictions - {base_name}', fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig(os.path.join(config.OUTPUT_DIR, "predictions", f"{base_name}_prediction.png"), 
                dpi=150, bbox_inches='tight')
    plt.show()

print(f"\n✅ Predictions saved to: {os.path.join(config.OUTPUT_DIR, 'predictions')}")